# MiniLM y tres cabezas — DataFrames y entrenamiento CUDA

Primero se muestran los DataFrames. Después se carga MiniLM y se entrenan tres cabezas, cada una con su DataLoader, target y pérdida. Ambas arquitecturas utilizan la RTX 4070 Ti SUPER y afinan el encoder compartido. No se utilizan máscaras de etiquetas ni vectores de tres targets.

FAQ: títulos de Stack Exchange, es_faq=1 según el criterio del proyecto; comentarios como es_faq=0. Logro y sentimiento: respuestas originales de Gemini. Los resultados nuevos se guardan en artifacts_v2/; los resultados anteriores se conservan como históricos.


In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT/'data/corpus_original.jsonl').exists(): ROOT = ROOT/'2026-10-04'
corpus = sorted([json.loads(s) for s in (ROOT/'data/corpus_original.jsonl').open(encoding='utf-8')], key=lambda r:r['id'])
id_map = {r['id']:i for i,r in enumerate(corpus,1)}
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', None)
print('Registros originales:',len(corpus))


Registros originales: 10000


## 1. df_faq: títulos de preguntas con es_faq=1
El mensaje positivo es exclusivamente el título, sin el cuerpo de la pregunta. Primero se muestran los positivos y después los comentarios es_faq=0 que completan el DataFrame de esta tarea.


In [2]:
df_faq = pd.DataFrame([
    dict(id_mensaje=id_map[r['id']], id_original=r['id'], mensaje=r['titulo'], es_faq=1,
         split=r['split'], split_group=r['split_group'], duplicate_of=r.get('duplicate_of'), url=r['url'])
    for r in corpus if r['source_type']=='question'
])
df_faq['id_mensaje']=df_faq.id_mensaje.astype('int64')
assert df_faq.mensaje.str.strip().ne('').all()
print('df_faq:',len(df_faq),'títulos; todos es_faq=1')
display(df_faq[['id_mensaje','mensaje','es_faq','split']].head(20))
display(df_faq.groupby('split').size().rename('títulos'))
df_faq_negativos = pd.DataFrame([
    dict(id_mensaje=id_map[r['id']], id_original=r['id'], mensaje=r['texto'], es_faq=0,
         split=r['split'], split_group=r['split_group'], duplicate_of=r.get('duplicate_of'), url=r['url'])
    for r in corpus if r['source_type']=='comment'
])
df_faq_entrenamiento = pd.concat([df_faq,df_faq_negativos],ignore_index=True).sort_values('id_mensaje').reset_index(drop=True)
print('Negativos FAQ:')
display(df_faq_negativos[['id_mensaje','mensaje','es_faq','split']].head(10))
display(df_faq_entrenamiento.es_faq.value_counts().sort_index().rename('registros'))


df_faq: 5000 títulos; todos es_faq=1


,id_mensaje,mensaje,es_faq,split
0,2501,Mostrar +1 (1 visto) correctamente con localStorage al reproducir video,1,train
1,2502,¿Como obtener los hijos inmediatos utilizando querySelectorAll?,1,train
2,2503,error en android studio en activity,1,test
3,2504,Escapar caracteres especiales,1,train
4,2505,¿Como desactivar el scroll vertical de un DIV?,1,train
5,2506,.d.ts en express no me toma los nuevos campos,1,train
6,2507,¿Cómo conectar Chat LM Studio con Django?,1,train
7,2508,¿Cómo eliminar los ultimos caracteres de una cadena?,1,train
8,2509,¿Como mantener centrados los elementos en kotlin mediante xml?,1,test
9,2510,Separ un registro que tiene un rango de fechas a un registro por dia sql,1,train


split
test           485
train         4047
validation     468
Name: títulos, dtype: int64

Negativos FAQ:


,id_mensaje,mensaje,es_faq,split
0,1,Y probaste instalar los paquetes faltantes??,0,test
1,2,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",0,train
2,3,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",0,train
3,4,"Si lo armas de una vez, tal vez llegue; inténtalo y también haz un var_dump(). No es pecado capital llamar al mismo archivo, se puede manejar sin problemas, aunque hasta el momento, tu código parece absurdo al hacer en el servidor algo que puede hacerse en el cliente, supongo -con optimismo- que librado el obstáculo hará algo más interesante.",0,train
4,5,Pero ¿por qué armas la sentencia en una cadena? si estas en t-sql mejor ejecuta directamente la sentencia parametrizando los valores. ¿El archivo binario esta en el servidor?.,0,train
5,6,"posiblemente no agregaste una implementación de la librería jaxb, compártenos el pom.xml o equivalente",0,train
6,7,con ese batch espero soluciones tu problema,0,train
7,8,"Tal y como está redactada, tu respuesta no es clara. Por favor, pulsa en edit para añadir detalles adicionales que ayuden a los demás a entender cómo esto responde a la pregunta formulada. Puedes encontrar más información sobre cómo escribir buenas respuestas en el centro de ayuda.",0,train
8,9,"@Sal los parámetros de OPENROWSET no pueden ser variables. Por lo tanto, necesita código dinámico para definir la ruta del archivo.",0,train
9,10,"Gracias a tu respuesta, vi la enorme contradicción que tenía la mía, gracias... totales!!!.",0,test


es_faq
0    5000
1    5000
Name: registros, dtype: int64

## 2. df_logro: comentarios y respuestas Gemini
Se lee annotation.es_logro directamente. True se representa como 1 y False como 0. Se conservan las banderas originales de calidad y duplicados para revisión, sin filtrar silenciosamente.


In [3]:
gemini = [r for r in corpus if (r.get('annotation') or {}).get('annotation_method')=='gemini_silver']
def common(r):
    return dict(id_mensaje=id_map[r['id']],id_original=r['id'],mensaje=r['texto'],
                split=r['split'],split_group=r['split_group'],url=r['url'],
                duplicate_of=r.get('duplicate_of'),eligible_silver_training=r.get('eligible_silver_training'),
                confianza_gemini=r['annotation'].get('confianza'))
df_logro = pd.DataFrame([
    dict(**common(r),es_logro_gemini=r['annotation']['es_logro'],
         es_logro=int(r['annotation']['es_logro']),evidencia_gemini=r['annotation'].get('evidencia'))
    for r in gemini
])
df_logro['id_mensaje']=df_logro.id_mensaje.astype('int64')
print('df_logro:',len(df_logro),'comentarios etiquetados por Gemini')
display(df_logro.es_logro.value_counts().sort_index().rename('comentarios'))
for label in [1,0]:
    print('Ejemplos es_logro =',label)
    display(df_logro[df_logro.es_logro.eq(label)][['id_mensaje','mensaje','es_logro_gemini','es_logro','split']].head(10))


df_logro: 5000 comentarios etiquetados por Gemini


es_logro
0    4752
1     248
Name: comentarios, dtype: int64

Ejemplos es_logro = 1


,id_mensaje,mensaje,es_logro_gemini,es_logro,split
22,23,"Actualicé la respuesta, tal vez mejore.",True,1,train
30,31,"Disculpad que no haya respondido antes, al final opte por emplear un proyecto que sí tenía bien la importación de las librerías, de todos modos me suele dar bastantes quebraderos de cabeza la importación de esas librerías. Mil gracias",True,1,train
31,32,muchas gracias por el aporte. Da solución a mi problema.,True,1,train
32,33,muchas gracias por el aporte. Da solución a mi problema.,True,1,train
33,34,muchas gracias por el aporte. Da solución a mi problema.,True,1,train
57,58,"Wee muchas gracias, pero lo que hice es usar un solo archivo html y hacer el login normal",True,1,validation
100,101,"Muchas gracias, justo ese era mi problema, solo como duda, con respecto al MutationObserver, se tiene que desconectar o se elimina automáticamente el observador?, ya que mediante esa función agrego el contenido de los select y si lo desconecto ya no me rellena los que se agreguen después del primero.",True,1,train
125,126,"Hola, me toco crear una nueva columna llamada Mes 1 y copiar los datos de la columna mes y realizar el proceso de Ordenar por columna entre los campos Mes1 y Numero de Mes y así, si me sirvió, saludos!.",True,1,train
127,128,"Mira How to Ask para que tu pregunta sea mejor recibida. También, aprovecha y haz el tour para entender mejor cómo funcionamos y de paso obtener tu primera medalla! Luego de que me pasara esto varias veces, decidi tirar xampp a la basura y usar docker para levantar un apache y un mysql y se acabaron todos mis problemas. que raro que no sepas tu SO.. no sabes si usas windows o linux?",True,1,train
153,154,"@A.Cedano Buenos dias! Disculpa la tardanza estos he estado fuera de la oficina y eso me ha ayudado a dar con la clave, no entraban los datos por que se pasaban mas datos en el json de los que se esperaban en la BBDD, filtre para que esos campos extras no entrasen y ya se solucionó el problema, mil gracias por tu respuesta!",True,1,train


Ejemplos es_logro = 0


,id_mensaje,mensaje,es_logro_gemini,es_logro,split
0,1,Y probaste instalar los paquetes faltantes??,False,0,test
1,2,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",False,0,train
2,3,"Estás poniendo solo partes del código que no contienen errores. Tampoco pones el error real. Sin esa información, no podemos ayudarte.",False,0,train
3,4,"Si lo armas de una vez, tal vez llegue; inténtalo y también haz un var_dump(). No es pecado capital llamar al mismo archivo, se puede manejar sin problemas, aunque hasta el momento, tu código parece absurdo al hacer en el servidor algo que puede hacerse en el cliente, supongo -con optimismo- que librado el obstáculo hará algo más interesante.",False,0,train
4,5,Pero ¿por qué armas la sentencia en una cadena? si estas en t-sql mejor ejecuta directamente la sentencia parametrizando los valores. ¿El archivo binario esta en el servidor?.,False,0,train
5,6,"posiblemente no agregaste una implementación de la librería jaxb, compártenos el pom.xml o equivalente",False,0,train
6,7,con ese batch espero soluciones tu problema,False,0,train
7,8,"Tal y como está redactada, tu respuesta no es clara. Por favor, pulsa en edit para añadir detalles adicionales que ayuden a los demás a entender cómo esto responde a la pregunta formulada. Puedes encontrar más información sobre cómo escribir buenas respuestas en el centro de ayuda.",False,0,train
8,9,"@Sal los parámetros de OPENROWSET no pueden ser variables. Por lo tanto, necesita código dinámico para definir la ruta del archivo.",False,0,train
9,10,"Gracias a tu respuesta, vi la enorme contradicción que tenía la mía, gracias... totales!!!.",False,0,test


## 3. df_sentimiento: comentarios y respuestas Gemini
Conversión acordada: negativo=0, neutro=0.5, mixto=0.5, positivo=1. Se conservan las 5,000 respuestas originales. El único indeterminado queda fuera de entrenamiento y se muestra para revisión.

df_sentimiento_entrenamiento contiene únicamente los registros con conversión acordada. Esta es una separación de datos de esta tarea, sin pérdidas enmascaradas.


In [4]:
sentiment_map={'negativo':0.0,'neutro':0.5,'mixto':0.5,'positivo':1.0}
df_sentimiento=pd.DataFrame([
    dict(**common(r),sentimiento_gemini=r['annotation']['sentimiento']) for r in gemini
])
df_sentimiento['id_mensaje']=df_sentimiento.id_mensaje.astype('int64')
df_sentimiento['sentimiento']=df_sentimiento.sentimiento_gemini.map(sentiment_map)
df_sentimiento_entrenamiento=df_sentimiento[df_sentimiento.sentimiento.notna()].copy()
df_sentimiento_revision=df_sentimiento[df_sentimiento.sentimiento.isna()].copy()
print('Respuestas originales Gemini:',len(df_sentimiento))
display(df_sentimiento.sentimiento_gemini.value_counts().rename('comentarios'))
for label,value in sentiment_map.items():
    print(label,'->',value)
    display(df_sentimiento_entrenamiento[df_sentimiento_entrenamiento.sentimiento_gemini.eq(label)]
            [['id_mensaje','mensaje','sentimiento_gemini','sentimiento','split']].head(5))
print('Respuestas pendientes de revisión:')
display(df_sentimiento_revision[['id_mensaje','mensaje','sentimiento_gemini','split']])


Respuestas originales Gemini: 5000


sentimiento_gemini
neutro           4114
positivo          530
negativo          344
mixto              11
indeterminado       1
Name: comentarios, dtype: int64

negativo -> 0.0


,id_mensaje,mensaje,sentimiento_gemini,sentimiento,split
1,2,"Varias cosas.. Esto no es una red social. Arrobar a alguien que no participa en la publicación no sirve de nada porque no seran notificados. Despues, creo que tenes un error de concepto terrible, tu archivo se esta llamando a si mismo, con una llamada asincronica, que es basicamente que estas llamando a otro archivo? no tiene mucho sentido lo que estas planteando..",negativo,0.0,train
40,41,"De todas formas, no está bien orientado el control de errores de código. Aunque es muy tradicional es mala práctica poner el ""or die"" en cualquier parte.",negativo,0.0,test
58,59,"Gracias nuevamente, pero sigo con el mismo problema, ya lo modifique de varias formas y sigue igual, no se si pensar que este mal mi lógica.",negativo,0.0,train
110,111,Lamento informarte que la empresa donde estás tiene pocos meses antes que apaguen la autenticación por usuario y contraseña para SMTP. Aquí un resumen de lo preguntado en Perplexity: perplexity.ai/search/… Saludos,negativo,0.0,train
112,113,la quiero guardar aqui: blumesportscenter.com/public/facturas/61.pdf pero no lo consigo @FranciscoIALover,negativo,0.0,train


neutro -> 0.5


...

,id_mensaje,mensaje,sentimiento_gemini,sentimiento,split
390,391,"lo demás esta bien explicado y atiende a los errores, pero me hubiera gustado algo mas completo que oriente hacia buenas practicas. XD",mixto,0.5,train
679,680,"@FranciscoIALover Gracias por la ayuda ! Quite el puerto :443, reedireccione al puerto 443 <VirtualHost *:80> ServerName lucasconde.ddns.net Redirect permanent / lucasconde.ddns.net </VirtualHost> y el SSL lo hice con Let's Encrypt que creo que es una authority (desde mi ignorancia) y sigo en la misma :/",mixto,0.5,train
913,914,"Listo, el copy del archivo ya funciona, tuve que cambiar into(""$buildDir/libs/config"") por into(""${buildDir}/libs/config""), pero el zip no se creo.",mixto,0.5,train
1397,1398,"Adapté tu propuesta a mi código y funciona, el mensaje se envia sin problemas pero en la terminal obtengo la siguiente excepción: Unhandled exception in asyncio atexit callback <function Browser.create.<locals>.browser_atexit at 0x0000024FD0500F40>: Executor shutdown has been called. Intenté hacer un await browser.close() para controlar la excepción.",mixto,0.5,train
1707,1708,"Siempre he sostenido que los genios pueden navegar en el desorden, analizando con más atención tu código, puedo decir dos cosas: 1) es una implementación horrible!!, realiza cientos de miles de iteraciones innecesarias, 2) la lógica detrás de esta... es impecable!!!!!!!, entonces, si has respaldado tu implementación, con una lógica que ""elimina"" los grupos no válidos, y esta te dice que hay 3.162.510 de los válidos, porque te empeñas en contradecirte!!! creando una ""demostración"" matemática sin fundamento?.",mixto,0.5,train


positivo -> 1.0


,id_mensaje,mensaje,sentimiento_gemini,sentimiento,split
9,10,"Gracias a tu respuesta, vi la enorme contradicción que tenía la mía, gracias... totales!!!.",positivo,1.0,test
24,25,"ok, muchisimas gracias",positivo,1.0,train
30,31,"Disculpad que no haya respondido antes, al final opte por emplear un proyecto que sí tenía bien la importación de las librerías, de todos modos me suele dar bastantes quebraderos de cabeza la importación de esas librerías. Mil gracias",positivo,1.0,train
31,32,muchas gracias por el aporte. Da solución a mi problema.,positivo,1.0,train
32,33,muchas gracias por el aporte. Da solución a mi problema.,positivo,1.0,train


Respuestas pendientes de revisión:


,id_mensaje,mensaje,sentimiento_gemini,split
4930,7431,@ScroogeMcDuck I imagine what,indeterminado,train


## Resumen antes del entrenamiento
Hasta aquí se muestran los registros originales. En la preparación de entrenamiento se excluyen los duplicados marcados y se conserva una sola aparición de cada texto por tarea, respetando la primera partición original. Se muestran los recuentos finales antes de entrenar. No se filtra por la bandera eligible_silver_training.


In [5]:
display(pd.DataFrame([
    {'DataFrame':'df_faq_entrenamiento','registros':len(df_faq_entrenamiento),'etiqueta':'títulos=1; comentarios=0'},
    {'DataFrame':'df_logro','registros':len(df_logro),'etiqueta':'Gemini es_logro 0/1'},
    {'DataFrame':'df_sentimiento_entrenamiento','registros':len(df_sentimiento_entrenamiento),'etiqueta':'Gemini sentimiento 0/0.5/1'},
    {'DataFrame':'df_sentimiento_revision','registros':len(df_sentimiento_revision),'etiqueta':'indeterminado; pendiente'}
]))


,DataFrame,registros,etiqueta
0,df_faq_entrenamiento,10000,títulos=1; comentarios=0
1,df_logro,5000,Gemini es_logro 0/1
2,df_sentimiento_entrenamiento,4999,Gemini sentimiento 0/0.5/1
3,df_sentimiento_revision,1,indeterminado; pendiente


## Hardware, datos en RAM y targets separados
CUDA es obligatoria para las dos arquitecturas; se comprueba RTX 4070 Ti SUPER. Se usan 8 hilos CPU para preparación, tokenización en RAM y memoria fijada para transferencias CUDA. Batch 128 y AMP FP16. La RAM disponible se registra: no es necesario ocupar 64 GB para este corpus.

Cada DataLoader tiene un solo target: es_faq, es_logro o sentimiento. El encoder es compartido; las tres cabezas tienen pesos independientes. Cada paso calcula únicamente la pérdida de su tarea, por lo que no se necesitan etiquetas de las otras dos tareas.


In [6]:
import os, random, time, hashlib
import numpy as np
import torch
from torch import nn
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score
from runtime import (hardware, task_frames, clean_frames, save_frames, tokenize_in_ram,
                     task_loader, to_cuda, predict_task, task_metrics, Monitor,
                     evaluate_variant, finalize_reports, Wrapper, TASKS, ARTIFACTS, SEED)
display(hardware())
frames = clean_frames({'es_faq':df_faq_entrenamiento,
                       'es_logro':df_logro,
                       'sentimiento':df_sentimiento_entrenamiento})
save_frames(frames,df_sentimiento_revision,mixed_value=0.5)
display(pd.DataFrame([
    {'tarea':task,'split':split,'registros':len(group),'targets':str(group[task].value_counts().sort_index().to_dict())}
    for task,df in frames.items() for split,group in df.groupby('split')
]))


{'gpu': 'NVIDIA GeForce RTX 4070 Ti SUPER',
 'vram_gib': 15.99169921875,
 'cpu_threads': 16,
 'torch_cpu_threads': 8,
 'ram_gib': 63.66278076171875,
 'torch': '2.5.1+cu121',
 'cuda': '12.1',
 'cpu': 'AMD Ryzen 7 8700F'}

,tarea,split,registros,targets
0,es_faq,test,903,"{0: 418, 1: 485}"
1,es_faq,train,7899,"{0: 3852, 1: 4047}"
2,es_faq,validation,980,"{0: 512, 1: 468}"
3,es_logro,test,418,"{0: 400, 1: 18}"
4,es_logro,train,3852,"{0: 3646, 1: 206}"
5,es_logro,validation,512,"{0: 490, 1: 22}"
6,sentimiento,test,418,"{0.0: 23, 0.5: 346, 1.0: 49}"
7,sentimiento,train,3851,"{0.0: 281, 0.5: 3150, 1.0: 420}"
8,sentimiento,validation,512,"{0.0: 38, 0.5: 419, 1.0: 55}"


## Dos escenarios, tres cabezas por escenario
A: cada cabeza es Linear(384,1) seguida de sigmoide al inferir.
B: cada cabeza es Linear(384,16), ReLU, Linear(16,4), ReLU, Linear(4,1), sigmoide al inferir.

Se mantienen logits durante entrenamiento y se usa BCEWithLogitsLoss, que incluye la operación sigmoide de forma numéricamente estable. Las cabezas de sentimiento y logro no reciben el target FAQ.


In [7]:
class ThreeHeads(nn.Module):
    def __init__(self,kind,encoder_path=None):
        super().__init__(); self.kind=kind
        self.encoder=SentenceTransformer(str(encoder_path or ROOT/'base_encoder'),device='cpu',local_files_only=True)
        self.encoder.max_seq_length=128
        assert self.encoder.get_sentence_embedding_dimension()==384
        def head():
            if kind=='linear': return nn.Linear(384,1)
            if kind=='mlp': return nn.Sequential(nn.Linear(384,16),nn.ReLU(),nn.Linear(16,4),nn.ReLU(),nn.Linear(4,1))
            raise ValueError(kind)
        self.heads=nn.ModuleDict({k:head() for k in TASKS})
    def logits(self,features,task):
        return self.heads[task](self.encoder(features)['sentence_embedding']).squeeze(-1)
    def scores(self,features):
        embedding=self.encoder(features)['sentence_embedding']
        return {k:self.heads[k](embedding).squeeze(-1).sigmoid() for k in TASKS}

In [8]:
for architecture in ['linear','mlp']:
    preview=ThreeHeads(architecture).cuda()
    print(architecture, 'encoder y cabezas en',next(preview.parameters()).device)
    display(preview.heads)
    print('Parámetros por cabeza:',{k:sum(p.numel() for p in h.parameters()) for k,h in preview.heads.items()})
    del preview
    torch.cuda.empty_cache()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

linear encoder y cabezas en cuda:0


C:\Users\saul_\AppData\Local\Temp\ipykernel_21996\900888258.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert self.encoder.get_sentence_embedding_dimension()==384


ModuleDict(
  (es_faq): Linear(in_features=384, out_features=1, bias=True)
  (es_logro): Linear(in_features=384, out_features=1, bias=True)
  (sentimiento): Linear(in_features=384, out_features=1, bias=True)
)

Parámetros por cabeza: {'es_faq': 385, 'es_logro': 385, 'sentimiento': 385}


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

mlp encoder y cabezas en cuda:0


C:\Users\saul_\AppData\Local\Temp\ipykernel_21996\900888258.py:6: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert self.encoder.get_sentence_embedding_dimension()==384


ModuleDict(
  (es_faq): Sequential(
    (0): Linear(in_features=384, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
  (es_logro): Sequential(
    (0): Linear(in_features=384, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
  (sentimiento): Sequential(
    (0): Linear(in_features=384, out_features=16, bias=True)
    (1): ReLU()
    (2): Linear(in_features=16, out_features=4, bias=True)
    (3): ReLU()
    (4): Linear(in_features=4, out_features=1, bias=True)
  )
)

Parámetros por cabeza: {'es_faq': 6233, 'es_logro': 6233, 'sentimiento': 6233}


## Entrenamiento por tarea, sin máscaras de pérdida
El código se muestra completo. Se alternan los tres DataLoaders para que las tres tareas afinen el mismo encoder. En cada paso, solo la cabeza elegida tiene gradientes; las otras dos no se actualizan. AdamW conserva los parámetros del encoder compartido y los de las tres cabezas. Esto permite exportar un solo encoder con tres cabezas.

Ambos escenarios parten del mismo MiniLM original y semilla 42: cuatro épocas, batch 128, encoder lr=2e-5, cabezas lr=1e-3. Se selecciona el checkpoint por pérdida media de validación. Los umbrales de FAQ/logro se eligen con el mismo wrapper de ventanas que se usa en test.


In [9]:
def train_variant(kind,frames,epochs=4,batch_size=128):
    info=hardware(); print('CUDA para',kind,info,flush=True)
    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
    model=ThreeHeads(kind).cuda()
    prepared=tokenize_in_ram(model,frames)
    loaders={k:task_loader(prepared[k],'train',batch_size,True) for k in TASKS}
    optimizer=torch.optim.AdamW([{'params':model.encoder.parameters(),'lr':2e-5},
                                 {'params':model.heads.parameters(),'lr':1e-3}],weight_decay=.01)
    losses={k:nn.BCEWithLogitsLoss() for k in TASKS}
    scaler=torch.amp.GradScaler('cuda')
    out=ARTIFACTS/kind; out.mkdir(parents=True,exist_ok=True)
    history=[]; best=float('inf'); start=time.perf_counter()
    with Monitor() as monitor:
        for epoch in range(epochs):
            model.train(); epoch_loss={k:[] for k in TASKS}
            iterators={k:iter(v) for k,v in loaders.items()}
            # Tres flujos de entrenamiento: cada paso tiene un único target de una única cabeza.
            while iterators:
                for task in list(iterators):
                    try: batch=next(iterators[task])
                    except StopIteration:
                        del iterators[task]; continue
                    features,y=to_cuda(batch,prepared[task]['names'])
                    optimizer.zero_grad(set_to_none=True)
                    with torch.autocast('cuda',dtype=torch.float16):
                        loss=losses[task](model.logits(features,task).float(),y)
                    if not torch.isfinite(loss): raise RuntimeError(f'Pérdida no finita: {task}')
                    scaler.scale(loss).backward(); scaler.unscale_(optimizer)
                    nn.utils.clip_grad_norm_(model.parameters(),1.)
                    scaler.step(optimizer); scaler.update(); epoch_loss[task].append(float(loss))
            model.eval(); val_loss={}
            with torch.inference_mode():
                for task,item in prepared.items():
                    values=[]
                    for batch in task_loader(item,'validation',batch_size):
                        features,y=to_cuda(batch,item['names'])
                        values.append((float(losses[task](model.logits(features,task).float(),y)),len(y)))
                    val_loss[task]=sum(v*n for v,n in values)/sum(n for _,n in values)
            score=float(np.mean(list(val_loss.values())))
            entry=dict(epoch=epoch+1,train_loss={k:float(np.mean(v)) for k,v in epoch_loss.items()},validation_loss=val_loss)
            history.append(entry); print(kind,entry,flush=True)
            if score<best:
                best=score; torch.save(model.state_dict(),out/'best.pt')
    elapsed=time.perf_counter()-start; peak_vram=torch.cuda.max_memory_allocated()/2**20
    model.load_state_dict(torch.load(out/'best.pt',weights_only=True)); model.eval()
    thresholds={}; validation={}
    for task,df in frames.items():
        val=df[df.split.eq('validation')]; p=predict_task(model,val,task)
        if task!='sentimiento':
            thresholds[task]=float(max(np.arange(.05,.96,.05),key=lambda t:f1_score(val[task],p>=t,zero_division=0)))
        validation[task]=task_metrics(val,task,p,thresholds.get(task,.5))
    model.encoder.save(str(out/'encoder')); torch.save(model.heads.state_dict(),out/'heads.pt')
    digest={k:hashlib.sha256(df[['id_original','mensaje',k,'split']].to_json(orient='records').encode()).hexdigest() for k,df in frames.items()}
    config=dict(kind=kind,tasks=TASKS,thresholds=thresholds,max_seq_length=128,
                architecture='384->1->sigmoid' if kind=='linear' else '384->16->4->1->sigmoid; ReLU entre capas',dataset_sha256=digest)
    (out/'config.json').write_text(json.dumps(config,indent=2),encoding='utf-8')
    report=dict(kind=kind,hardware=info,epochs=epochs,batch_size=batch_size,history=history,validation=validation,
                train_seconds=elapsed,train_peak_ram_mb=monitor.peak/2**20,train_peak_vram_mb=peak_vram,
                head_parameters={k:sum(p.numel() for p in h.parameters()) for k,h in model.heads.items()},dataset_sha256=digest)
    (out/'training.json').write_text(json.dumps(report,indent=2),encoding='utf-8')
    del prepared,optimizer,model; torch.cuda.empty_cache(); return report

In [10]:
REENTRENAR = False
EPOCHS = 4
BATCH_SIZE = 128
training={}
for architecture in ['linear','mlp']:
    report_path=ARTIFACTS/architecture/'training.json'
    expected={k:hashlib.sha256(df[['id_original','mensaje',k,'split']].to_json(orient='records').encode()).hexdigest() for k,df in frames.items()}
    cached=json.loads(report_path.read_text(encoding='utf-8')) if report_path.exists() else None
    if REENTRENAR or cached is None or cached.get('dataset_sha256')!=expected:
        training[architecture]=train_variant(architecture,frames,epochs=EPOCHS,batch_size=BATCH_SIZE)
    else:
        training[architecture]=cached
display(pd.DataFrame([{k:v for k,v in report.items() if k in ['kind','epochs','batch_size','train_seconds','train_peak_ram_mb','train_peak_vram_mb']}
                      for report in training.values()]))
display({k:v['history'] for k,v in training.items()})


,kind,epochs,batch_size,train_seconds,train_peak_ram_mb,train_peak_vram_mb
0,linear,4,128,49.904185,2427.515625,4672.865234
1,mlp,4,128,49.060951,2723.609375,4671.729492


{'linear': [{'epoch': 1,
   'train_loss': {'es_faq': 0.20296856971277344,
    'es_logro': 0.28127989893959415,
    'sentimiento': 0.6843330725546806},
   'validation_loss': {'es_faq': 0.05826683506673696,
    'es_logro': 0.13062025979161263,
    'sentimiento': 0.6671732068061829}},
  {'epoch': 2,
   'train_loss': {'es_faq': 0.048160612966204365,
    'es_logro': 0.12503605732514012,
    'sentimiento': 0.6642614006996155},
   'validation_loss': {'es_faq': 0.05857734366567159,
    'es_logro': 0.1122600045055151,
    'sentimiento': 0.6612756550312042}},
  {'epoch': 3,
   'train_loss': {'es_faq': 0.025212827570466025,
    'es_logro': 0.07600888381561925,
    'sentimiento': 0.644977331161499},
   'validation_loss': {'es_faq': 0.05353883796808671,
    'es_logro': 0.10226688161492348,
    'sentimiento': 0.6806737631559372}},
  {'epoch': 4,
   'train_loss': {'es_faq': 0.012212821786599834,
    'es_logro': 0.040582393281041615,
    'sentimiento': 0.6411224353697992},
   'validation_loss': {'es_f

## F1, inferencia, RAM y VRAM
Se evalúa cada tarea en su propio test. F1 binario para FAQ/logro y F1 macro para sentimiento, con clases negativo/central(neutro o mixto)/positivo y cortes fijos 0.25/0.75. También MAE para sentimiento continuo.

Benchmark del wrapper completo: 100 mensajes mezclados, batch=1, calentamiento y sincronización CUDA. p50/p95, mensajes/s, RSS y VRAM asignada. Se mide en un proceso nuevo por arquitectura para comparar RAM.


In [11]:
import subprocess, sys
evaluation={}
for architecture in ['linear','mlp']:
    completed=subprocess.run([sys.executable,str(ROOT/'runtime.py'),'evaluate',architecture],capture_output=True,text=True,encoding='utf-8',errors='replace')
    if completed.returncode:
        raise RuntimeError(completed.stdout+'\n'+completed.stderr)
    evaluation[architecture]=json.loads((ARTIFACTS/architecture/'evaluation.json').read_text(encoding='utf-8'))
display(pd.DataFrame([{ 'arquitectura':architecture,'tarea':task,**metric}
                      for architecture,report in evaluation.items() for task,metric in report['test'].items()]))
display(pd.DataFrame({k:v['benchmark'] for k,v in evaluation.items()}).T)
comparison=finalize_reports()
display(comparison['recommendation'])


,arquitectura,tarea,f1,f1_macro,support,positives,threshold,mae,confusion_matrix,classes
0,linear,es_faq,0.977597,0.975449,903,485.0,0.75,NaN,NaN,NaN
1,linear,es_logro,0.400000,0.686891,418,18.0,0.20,NaN,NaN,NaN
2,linear,sentimiento,NaN,0.500032,418,NaN,NaN,0.116815,"[[0, 23, 0], [0, 345, 1], [0, 29, 20]]","[negativo, central_neutro_mixto, positivo]"
3,mlp,es_faq,0.980513,0.978824,903,485.0,0.50,NaN,NaN,NaN
4,mlp,es_logro,0.439024,0.705047,418,18.0,0.25,NaN,NaN,NaN
5,mlp,sentimiento,NaN,0.521744,418,NaN,NaN,0.103848,"[[0, 23, 0], [0, 328, 18], [0, 17, 32]]","[negativo, central_neutro_mixto, positivo]"


,ram_rss_peak_mb,vram_peak_allocated_mb,latency_p50_ms,latency_p95_ms,messages_per_second,samples,batch_size
linear,1414.910156,461.847168,5.29565,6.252510,183.795487,100.0,1.0
mlp,1424.453125,461.918945,5.50280,6.390035,177.604247,100.0,1.0


{'kind': 'mlp',
 'validation_macro_f1': {'linear': 0.7306678392296267,
  'mlp': 0.7752762235836572},
 'rule': 'MLP si mejora macro F1 medio de validation >0.01; en otro caso lineal. Test no selecciona. Una semilla.'}

## Wrapper y contrato
Un encoder y tres cabezas. Entrada {id_mensaje:int64, mensaje:string no vacío}; salida con el mismo ID y tres scores independientes es_faq/es_logro/sentimiento entre 0 y 1. Las probabilidades de tareas diferentes no suman uno. Sentimiento es un índice continuo.

Mensajes largos: ventanas de 128 tokens con solapamiento 16; promedio de los scores por ventana. Entrenamiento usa los primeros 128 tokens. Los resultados son exploratorios: una semilla y etiquetas Gemini, sin búsqueda de hiperparámetros.


In [12]:
service=Wrapper(ARTIFACTS/comparison['recommendation']['kind'])
examples=[{'id_mensaje':101,'mensaje':df_faq.iloc[0].mensaje},
          {'id_mensaje':102,'mensaje':df_logro[df_logro.es_logro.eq(1)].iloc[0].mensaje},
          {'id_mensaje':103,'mensaje':'Estoy contento con lo aprendido.'}]
display(pd.DataFrame([dict(mensaje=item['mensaje'],**service(item)) for item in examples]))


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

C:\Users\saul_\Documents\Codex\2026-10-04\runtime.py:94: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  assert self.encoder.get_sentence_embedding_dimension()==384


,mensaje,id_mensaje,es_faq,es_logro,sentimiento
0,Mostrar +1 (1 visto) correctamente con localStorage al reproducir video,101,0.995453,0.046989,0.432114
1,"Actualicé la respuesta, tal vez mejore.",102,0.003446,0.417303,0.856351
2,Estoy contento con lo aprendido.,103,0.011968,0.387337,0.909580
